# Herzkrankheitsdaten bereinigen und für Modelle vorbereiten

Dieses Notebook liest `data/heart_disease_uci.csv`, lässt die Originaldatei unverändert und exportiert bereinigte Daten nach `data/processed/`.

**Zielvariable:** Für die binäre Klassifikation verwenden wir `target = (num > 0)` (0: keine Erkrankung im Datensatz, 1: Erkrankung). Die ursprünglichen Schweregrade `num` (0–4) bleiben in der bereinigten Tabelle erhalten. Für eine spätere Mehrklassenklassifikation muss die Aufteilung entsprechend neu erstellt werden.

**Bereinigungsregeln:** Leere Angaben werden als fehlend behandelt. Nichtpositive Werte in `trestbps`, `chol` und `thalch` werden zu fehlenden Werten. Insbesondere `chol = 0` wird als fehlende Messung behandelt, nicht als echter Cholesterinwert. Negative `oldpeak`-Werte und hohe Messwerte werden nicht pauschal gelöscht. Die stark unvollständigen Merkmale `ca` und `thal` bleiben erhalten; ihre fehlenden Werte werden später explizit behandelt.

**Vorhersagen:** `id`, Herkunftsklinik `dataset` und ursprüngliches Label `num` sind keine Modellmerkmale. Imputation, Skalierung und Kategorienkodierung werden ausschließlich auf dem Trainingssatz angepasst. Die zufällige, nach Zielklasse geschichtete 80/20-Aufteilung prüft Vorhersagen innerhalb dieser gemischten Datenquelle; sie prüft nicht die Übertragbarkeit auf eine neue Klinik.


In [1]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
# Funktioniert aus dem Projektverzeichnis oder aus dessen data-Unterverzeichnis.
ROOT = Path.cwd()
if not (ROOT / "data/heart_disease_uci.csv").exists():
    ROOT = ROOT.parent
INPUT_PATH = ROOT / "data/heart_disease_uci.csv"
OUTPUT_DIR = ROOT / "data/processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

raw = pd.read_csv(INPUT_PATH, na_values=["?", "NA", "N/A", "null", "None"])
print(f"Originaldaten: {raw.shape[0]} Zeilen, {raw.shape[1]} Spalten")
display(raw.head())


Originaldaten: 920 Zeilen, 16 Spalten


,id,age,sex,dataset,cp,trestbps,chol,fbs,restecg,thalch,exang,oldpeak,slope,ca,thal,num
0,1,63,Male,Cleveland,typical angina,145.0,233.0,True,lv hypertrophy,150.0,False,2.3,downsloping,0.0,fixed defect,0
1,2,67,Male,Cleveland,asymptomatic,160.0,286.0,False,lv hypertrophy,108.0,True,1.5,flat,3.0,normal,2
2,3,67,Male,Cleveland,asymptomatic,120.0,229.0,False,lv hypertrophy,129.0,True,2.6,flat,2.0,reversable defect,1
3,4,37,Male,Cleveland,non-anginal,130.0,250.0,False,normal,187.0,False,3.5,downsloping,0.0,normal,0
4,5,41,Female,Cleveland,atypical angina,130.0,204.0,False,lv hypertrophy,172.0,False,1.4,upsloping,0.0,normal,0


## 1. Datentypen und Kategorien prüfen
Unbekannte Kategorien und ungültige Zielwerte lösen einen Fehler aus, damit geänderte Eingabedaten nicht unbemerkt falsch verarbeitet werden.


In [2]:
numeric_columns = ["age", "trestbps", "chol", "thalch", "oldpeak"]
categorical_columns = ["sex", "cp", "fbs", "restecg", "exang", "slope", "ca", "thal"]
expected_columns = ["id", "age", "sex", "dataset", "cp", "trestbps", "chol", "fbs",
                    "restecg", "thalch", "exang", "oldpeak", "slope", "ca", "thal", "num"]
if set(raw.columns) != set(expected_columns):
    raise ValueError(f"Unerwartete Spalten: {list(raw.columns)}")

clean = raw.copy(deep=True)
# Leerzeichen und Schreibweisen vereinheitlichen.
for column in clean.select_dtypes(include=["object", "string"]).columns:
    clean[column] = clean[column].astype("string").str.strip().replace("", pd.NA)

numeric_parse_failures = {}
for column in numeric_columns + ["id", "ca", "num"]:
    parsed = pd.to_numeric(clean[column], errors="coerce")
    numeric_parse_failures[column] = int((clean[column].notna() & parsed.isna()).sum())
    if column in ["id", "num"] and (parsed.isna().any() or (parsed % 1 != 0).any()):
        raise ValueError(f"{column} muss vollständig und ganzzahlig sein.")
    clean[column] = parsed.astype(float)

if not clean["num"].isin([0, 1, 2, 3, 4]).all():
    raise ValueError("num muss zwischen 0 und 4 liegen.")
if not clean["id"].is_unique:
    raise ValueError("Mehrfach verwendete IDs prüfen, bevor Patienten aufgeteilt werden.")
clean[["id", "num"]] = clean[["id", "num"]].astype(int)

# Boolesche Angaben werden als explizite Kategorien behandelt.
for column in ["fbs", "exang"]:
    values = clean[column].astype("string").str.lower()
    if not values.dropna().isin(["true", "false", "1", "0"]).all():
        raise ValueError(f"Unbekannte boolesche Werte in {column}.")
    clean[column] = values.map({"true": "true", "false": "false", "1": "true", "0": "false"})

clean["thal"] = clean["thal"].replace({"reversable defect": "reversible defect"})
allowed_categories = {
    "sex": ["Male", "Female"],
    "cp": ["typical angina", "atypical angina", "non-anginal", "asymptomatic"],
    "restecg": ["normal", "lv hypertrophy", "st-t abnormality"],
    "slope": ["upsloping", "flat", "downsloping"],
    "thal": ["normal", "fixed defect", "reversible defect"],
}
for column, allowed in allowed_categories.items():
    unknown = clean.loc[clean[column].notna() & ~clean[column].isin(allowed), column].unique()
    if len(unknown):
        raise ValueError(f"Unbekannte Kategorien in {column}: {unknown}")


## 2. Fehlende und ungültige Messwerte behandeln
Es werden noch keine Ersatzwerte aus dem Gesamtdatensatz berechnet. Die bereinigte CSV behält fehlende Angaben, sodass sie auch in anderen Modell-Pipelines nutzbar bleibt.


In [3]:
invalid_measurements = {}
for column in ["age", "trestbps", "chol", "thalch"]:
    invalid = clean[column].notna() & (clean[column] <= 0)
    invalid_measurements[column] = int(invalid.sum())
    clean.loc[invalid, column] = np.nan

invalid_ca = clean["ca"].notna() & ~clean["ca"].isin([0, 1, 2, 3])
invalid_measurements["ca"] = int(invalid_ca.sum())
clean.loc[invalid_ca, "ca"] = np.nan
# ca ist eine diskrete Kategorie; keine künstlichen Zwischenwerte imputieren.
clean["ca"] = clean["ca"].map({0: "0", 1: "1", 2: "2", 3: "3"})

# Identische Datensätze ohne technische ID nur einmal behalten.
duplicate_mask = clean.duplicated(subset=[c for c in expected_columns if c != "id"])
duplicates_removed = int(duplicate_mask.sum())
clean = clean.loc[~duplicate_mask].copy().reset_index(drop=True)
clean["target"] = (clean["num"] > 0).astype(int)
# sklearn erhält für alle fehlenden Kategorien einheitlich np.nan.
for column in categorical_columns:
    clean[column] = clean[column].astype(object).where(clean[column].notna(), np.nan)

missing_report = pd.DataFrame({
    "fehlend_original": raw.isna().sum(),
    "fehlend_bereinigt": clean[expected_columns].isna().sum(),
    "fehlend_bereinigt_prozent": (clean[expected_columns].isna().mean() * 100).round(2),
})
display(missing_report)
print("Ungültige Messwerte zu fehlend:", invalid_measurements)
print("Entfernte Duplikate:", duplicates_removed)
display(clean["target"].value_counts().sort_index().rename("Anzahl"))


Ungültige Messwerte zu fehlend: {'age': 0, 'trestbps': 1, 'chol': 172, 'thalch': 0, 'ca': 0}
Entfernte Duplikate: 2


,fehlend_original,fehlend_bereinigt,fehlend_bereinigt_prozent
id,0,0,0.00
age,0,0,0.00
sex,0,0,0.00
dataset,0,0,0.00
cp,0,0,0.00
trestbps,59,60,6.54
chol,30,201,21.90
fbs,90,90,9.80
restecg,2,2,0.22
thalch,55,55,5.99


target
0    410
1    508
Name: Anzahl, dtype: int64

## 3. Trainings- und Testdaten trennen
Die Patient-IDs dienen nur zur Rückverfolgung. Der Testdatensatz bleibt bei allen lernenden Vorverarbeitungsschritten außen vor.


In [4]:
feature_columns = numeric_columns + categorical_columns
X = clean[feature_columns].copy()
y = clean["target"].copy()
train_index, test_index = train_test_split(
    clean.index, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
X_train, X_test = X.loc[train_index].copy(), X.loc[test_index].copy()
y_train, y_test = y.loc[train_index].copy(), y.loc[test_index].copy()
print(f"Training: {len(X_train)}, Test: {len(X_test)}")
display(pd.DataFrame({"Training": y_train.value_counts(), "Test": y_test.value_counts()}).sort_index())


Training: 734, Test: 184


,Training,Test
target,,
0,328,82
1,406,102


## 4. Modellfähige Merkmale erstellen
Numerische Lücken werden durch Trainingsmediane ersetzt; zusätzliche Indikatoren markieren fehlende Messungen. Numerische Merkmale werden standardisiert. Fehlende Kategorien erhalten `missing`, anschließend werden Kategorien per One-Hot-Encoding kodiert. Unbekannte Kategorien bei neuen Daten werden toleriert.

Bei Kreuzvalidierung muss ein **frischer** Preprocessor gemeinsam mit dem Modell in eine Pipeline, damit jeder Fold seine eigenen Ersatzwerte lernt.


In [5]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median", add_indicator=True, keep_empty_features=True)),
    ("scaler", StandardScaler()),
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="missing", keep_empty_features=True)),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_columns),
    ("categorical", categorical_pipeline, categorical_columns),
])
train_values = preprocessor.fit_transform(X_train)
test_values = preprocessor.transform(X_test)
encoded_features = preprocessor.get_feature_names_out()
X_train_ready = pd.DataFrame(train_values, columns=encoded_features, index=X_train.index)
X_test_ready = pd.DataFrame(test_values, columns=encoded_features, index=X_test.index)
display(X_train_ready.head())
print(f"Modellmerkmale: {len(encoded_features)}")


Modellmerkmale: 37


,numeric__age,numeric__trestbps,numeric__chol,numeric__thalch,numeric__oldpeak,numeric__missingindicator_trestbps,numeric__missingindicator_chol,numeric__missingindicator_thalch,numeric__missingindicator_oldpeak,categorical__sex_Female,categorical__sex_Male,categorical__cp_asymptomatic,categorical__cp_atypical angina,categorical__cp_non-anginal,categorical__cp_typical angina,categorical__fbs_false,categorical__fbs_missing,categorical__fbs_true,categorical__restecg_lv hypertrophy,categorical__restecg_normal,categorical__restecg_st-t abnormality,categorical__exang_false,categorical__exang_missing,categorical__exang_true,categorical__slope_downsloping,categorical__slope_flat,categorical__slope_missing,categorical__slope_upsloping,categorical__ca_0,categorical__ca_1,categorical__ca_2,categorical__ca_3,categorical__ca_missing,categorical__thal_fixed defect,categorical__thal_missing,categorical__thal_normal,categorical__thal_reversible defect
625,-0.275038,-0.676985,-0.108875,-0.429768,0.606406,-0.270369,1.886533,-0.26156,-0.273259,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
352,-1.346579,-0.676985,0.905502,1.276224,-0.809313,-0.270369,-0.530073,-0.26156,-0.273259,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
97,0.689348,0.994790,0.223103,0.760459,1.644601,-0.270369,-0.530073,-0.26156,-0.273259,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
518,0.046424,-0.398356,-0.551513,0.085997,-0.809313,-0.270369,-0.530073,-0.26156,-0.273259,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
246,0.475040,-1.791502,-0.219535,0.720785,-0.714932,-0.270369,-0.530073,-0.26156,-0.273259,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


## 5. Ergebnisse prüfen und exportieren
`heart_disease_clean.csv` enthält alle bereinigten Daten einschließlich ursprünglichem Label und IDs. Die Dateien `train_clean.csv` / `test_clean.csv` enthalten die aufgeteilten, noch nicht imputierten Daten. Die Dateien `train_model_ready.csv` / `test_model_ready.csv` enthalten numerische Merkmale und `target`; beide sind direkt für Modelle nutzbar.


In [6]:
assert set(train_index).isdisjoint(set(test_index))
assert len(train_index) + len(test_index) == len(clean)
assert clean["id"].is_unique
assert not {"id", "dataset", "num", "target"}.intersection(feature_columns)
assert np.isfinite(train_values).all() and np.isfinite(test_values).all()
assert X_train_ready.columns.equals(X_test_ready.columns)
assert not X_train_ready.isna().any().any()
assert not X_test_ready.isna().any().any()
# Kontrolliert, dass nur Trainingsmediane verwendet wurden.
np.testing.assert_allclose(
    preprocessor.named_transformers_["numeric"].named_steps["imputer"].statistics_,
    X_train[numeric_columns].median().fillna(0).to_numpy(),
)

clean.to_csv(OUTPUT_DIR / "heart_disease_clean.csv", index=False)
missing_report.to_csv(OUTPUT_DIR / "missing_values_report.csv", index_label="feature")
clean.loc[train_index].to_csv(OUTPUT_DIR / "train_clean.csv", index=False)
clean.loc[test_index].to_csv(OUTPUT_DIR / "test_clean.csv", index=False)
X_train_ready.assign(target=y_train).to_csv(OUTPUT_DIR / "train_model_ready.csv", index=False)
X_test_ready.assign(target=y_test).to_csv(OUTPUT_DIR / "test_model_ready.csv", index=False)
joblib.dump(preprocessor, OUTPUT_DIR / "preprocessor.joblib")

report = {
    "input_rows": len(raw), "clean_rows": len(clean),
    "duplicates_removed": duplicates_removed,
    "numeric_parse_failures": numeric_parse_failures,
    "invalid_measurements_set_missing": invalid_measurements,
    "target_definition": "1 if num > 0 else 0",
    "excluded_model_columns": ["id", "dataset", "num"],
    "train_rows": len(X_train), "test_rows": len(X_test),
    "random_state": RANDOM_STATE, "feature_columns": feature_columns,
    "encoded_features": encoded_features.tolist(),
    "train_patient_ids": clean.loc[train_index, "id"].tolist(),
    "test_patient_ids": clean.loc[test_index, "id"].tolist(),
}
(OUTPUT_DIR / "cleaning_report.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
print(f"Alle Prüfungen bestanden. Ergebnisse: {OUTPUT_DIR}")


Alle Prüfungen bestanden. Ergebnisse: /Users/felix/Desktop/Code/ML/Heart_Thing/data/processed


## Weiterverwenden für Vorhersagen

Mit den numerischen Tabellen kannst du ein Modell auf `X_train_ready` und `y_train` trainieren und einmalig auf `X_test_ready` und `y_test` bewerten. `target` darf nicht als Eingabemerkmal verwendet werden.

Für Modellauswahl mit Kreuzvalidierung nutze die untransformierten `X_train` und `y_train` und eine gemeinsame Pipeline, zum Beispiel:

```python
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

model = Pipeline([
    ("preprocessing", clone(preprocessor)),
    ("classifier", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
])
scores = cross_val_score(model, X_train, y_train, cv=5, scoring="roc_auc")
model.fit(X_train, y_train)
predictions = model.predict(X_test)
```

Neue Rohdaten müssen dieselben Bereinigungsregeln und Merkmalsnamen verwenden. Der gespeicherte `preprocessor.joblib` erwartet bereits bereinigte Merkmale; für neue Daten nur `transform()` aufrufen. Eine gemeinsame trainierte Modell-Pipeline kann direkt `predict()` nutzen.
